In [ ]:
import pandas as pd
train_features = pd.read_csv("/kaggle/input/lish-moa/train_features.csv")
train_targets = pd.read_csv("/kaggle/input/lish-moa/train_targets_scored.csv")
train_drugs = pd.read_csv("/kaggle/input/lish-moa/train_drug.csv")
test_features = pd.read_csv("/kaggle/input/lish-moa/test_features.csv")
sample_submission = pd.read_csv("/kaggle/input/lish-moa/sample_submission.csv")

X_train = train_features.merge(train_drugs, on="sig_id", how="left") # extra column  "drug_id"
y_train = train_targets
X_test = test_features
X_train['cp_type'] = X_train['cp_type'].astype('category')
X_train['cp_dose'] = X_train['cp_dose'].astype('category')
X_train['cp_time'] = X_train['cp_time'].astype('category')
X_train['drug_id'] = X_train['drug_id'].astype('category')
drug_counts = X_train['drug_id'].value_counts()

# Decision tree - Valid

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, StratifiedKFold 
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import log_loss
from itertools import product

# evaluation metric and loss function
def competition_log_loss(y_true, y_pred):
    eps = 1e-15
    y_pred = np.clip(y_pred, eps, 1 - eps)
    losses = []
    
    for i in range(y_true.shape[0]):
        loss = log_loss(y_true.iloc[i, :], y_pred[i, :], labels=[0, 1])
        losses.append(loss)
    
    return np.mean(losses)



def generate_submission(model_pipelines, X_test, sample_submission, TARGET_COLS):
    
    is_compound_test = X_test['cp_type'] != 'ctl_vehicle'
    X_test_cp = X_test[is_compound_test].drop(columns=['sig_id'])

    num_targets = len(TARGET_COLS)
    final_y_test_pred_sum = np.zeros((X_test_cp.shape[0], num_targets))
    
    for model_pipeline in model_pipelines:
        X_test_proc = model_pipeline.named_steps['preprocessor'].transform(X_test_cp)

        y_test_pred_cp = np.zeros((X_test_cp.shape[0], num_targets))
        classifier_estimators = model_pipeline.named_steps['classifier'].estimators_
        
        for i, est in enumerate(classifier_estimators):
            y_test_pred_cp[:, i] = safe_predict_proba_class_1(est, X_test_proc)
        
        final_y_test_pred_sum += y_test_pred_cp

    y_test_pred_final = final_y_test_pred_sum / len(model_pipelines)

    submission = sample_submission.copy()
    submission.loc[:, TARGET_COLS] = 0.0

    compound_sig_ids = X_test[is_compound_test]['sig_id'].values
    submission_indices = submission[submission['sig_id'].isin(compound_sig_ids)].index
    submission.loc[submission_indices, TARGET_COLS] = y_test_pred_final
    
    return submission















def safe_predict_proba_class_1(estimator, X):
    """
    Safely predicts probability of class 1 (p=1), handling degenerate classifiers 
    that were only trained on one class (e.g., all 0s in a sparse MoA target).
    """
    probas = estimator.predict_proba(X)
    
    # Check if the classifier produced two probability columns (binary case)
    if probas.shape[1] == 2:
        
        return probas[:, 1]
    else:
        # If probas.shape[1] == 1, the classifier is degenerate. 
        # Since MoA targets are sparse (mostly 0s), we assume p(class=1) is 0.0.
        return np.zeros(X.shape[0])



# Data Loading and Preparation
TARGET_COLS = [col for col in y_train.columns if col != 'sig_id']
ALL_FEATURES = X_train.columns.tolist()
NUM_FEATURES = [col for col in ALL_FEATURES if col.startswith('g-') or col.startswith('c-')]
CAT_FEATURES = ['cp_type', 'cp_time', 'cp_dose']
FEATURES = NUM_FEATURES + CAT_FEATURES


# Filter out control samples from the training set, as they have no MoAs.
is_compound_train = X_train['cp_type'] != 'ctl_vehicle'
X_train_filtered = X_train[is_compound_train].reset_index(drop=True)
y_train_filtered = y_train[is_compound_train].drop(columns=['sig_id']).reset_index(drop=True)

# save DRUG_ID and MOA_COUNT_TARGET and COMBINED_CV_TARGET  before dropping from features
DRUG_ID = X_train_filtered['drug_id']

X_train_filtered['moa_count'] = y_train_filtered.sum(axis=1)
MOA_COUNT_TARGET = X_train_filtered['moa_count']

X_train_filtered['combined_cv_target'] = DRUG_ID.astype(str) + '_' + MOA_COUNT_TARGET.astype(str)
COMBINED_CV_TARGET = X_train_filtered['combined_cv_target']

# Drop columns not needed for training features
X_train_filtered = X_train_filtered.drop(columns=['sig_id', 'moa_count', 'combined_cv_target', 'drug_id'])

# cross validation configuration
N_SPLITS = 5


# define CV strategies
gkf = GroupKFold(n_splits=N_SPLITS) # CV 1: by drug_id (pure grouping)
skf_moa = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42) # CV 2: by BINNED MoA count (pure stratification)
skf_combined = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42) # CV 3: by Combined Target (simulated combined CV)

cv_configs = [
    {'name': 'Drug GroupKFold', 'cv_object': gkf, 'groups': DRUG_ID, 'stratify_target': None},
    {'name': 'MoA StratifiedKFold (Binned)', 'cv_object': skf_moa, 'groups': None, 'stratify_target': MOA_COUNT_TARGET},
    {'name': 'Combined Drug+MoA Stratification', 'cv_object': skf_combined, 'groups': None, 'stratify_target': COMBINED_CV_TARGET}
]

# preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), NUM_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_FEATURES)
    ],
    remainder='drop' 
)

# versions of dt
param_grid = {
    'max_depth': [5, 10, 15],  
    'min_samples_leaf': [5, 10] 
}
param_names = list(param_grid.keys())
param_combinations = list(product(*param_grid.values()))

# store best parameters and losses from CV types
best_models = {}

# cross validation
for config in cv_configs:

    cv_name = config['name']
    cv_object = config['cv_object']
    groups = config['groups']
    stratify_target = config['stratify_target']
    
    
    current_best_loss = float('inf')
    current_best_params = None
    param_results = {}
    
    for params_tuple in param_combinations:
        current_params = dict(zip(param_names, params_tuple))
        
        fold_losses = []
        

        if cv_name == 'Drug GroupKFold':
            cv_splits = cv_object.split(X_train_filtered, y_train_filtered, groups=groups)
        else: 
            cv_splits = cv_object.split(X_train_filtered, stratify_target)


        # Iterate through cross-validation folds
        for fold, (train_index, val_index) in enumerate(cv_splits):
            
            # Split data for the current fold
            X_train_fold = X_train_filtered.iloc[train_index]
            X_valid_fold = X_train_filtered.iloc[val_index]
            y_train_fold = y_train_filtered.iloc[train_index]
            y_valid_fold = y_train_filtered.iloc[val_index]

            # Define model with current parameters
            current_tree = DecisionTreeClassifier(random_state=42, **current_params)
            current_multi_model = MultiOutputClassifier(current_tree, n_jobs=-1)
            #train
            current_pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor),
                ('classifier', current_multi_model)
            ])
            current_pipeline.fit(X_train_fold, y_train_fold)
            
            # Calculate Validation Loss
            X_valid_proc = current_pipeline.named_steps['preprocessor'].transform(X_valid_fold)
            y_valid_pred = np.zeros(y_valid_fold.shape)
            
            estimators_valid = current_pipeline.named_steps['classifier'].estimators_
            for i, est in enumerate(estimators_valid):
                y_valid_pred[:, i] = safe_predict_proba_class_1(est, X_valid_proc)
                
            val_loss = competition_log_loss(y_valid_fold, y_valid_pred)
            fold_losses.append(val_loss)

        # Calculate average loss for the current parameter set
        avg_val_loss = np.mean(fold_losses)
        param_results[params_tuple] = avg_val_loss
        
        # Check for best parameters for this CV type
        if avg_val_loss < current_best_loss:
            current_best_loss = avg_val_loss
            current_best_params = current_params

        print(f"  Model: {current_params} | Avg Val Loss: {avg_val_loss:.4f}")

    # Store and display the best result for this CV type
    best_models[cv_name] = {
        'best_params': current_best_params,
        'best_loss': current_best_loss
    }
    
    print(f"\n--- Best Model for {cv_name} ---")
    print(f"Best Loss: {current_best_loss:.4f}, Params: {current_best_params}")




# final Training on Full Data using the best models
final_pipelines = {}
best_params_all = {}

X_train_full = X_train_filtered

for cv_name, result in best_models.items():
    best_params = result['best_params']
    
    print(f"\nTraining FINAL model for {cv_name} on ALL training data...")
    
    # Create Model using best parameters
    final_tree = DecisionTreeClassifier(random_state=42, **best_params)
    final_multi_model = MultiOutputClassifier(final_tree, n_jobs=-1)

    # Create Final Pipeline
    final_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', final_multi_model)
    ])

    # Train on ALL available filtered data
    final_pipeline.fit(X_train_full, y_train_filtered)
    
    # Store the pipeline and parameters 
    final_pipelines[cv_name] = final_pipeline
    best_params_all[cv_name] = best_params
    
    # Calculate Training Loss for reference
    X_train_proc = final_pipeline.named_steps['preprocessor'].transform(X_train_full)
    y_train_pred = np.zeros(y_train_filtered.shape)
    

    estimators_train = final_pipeline.named_steps['classifier'].estimators_
    for i, est in enumerate(estimators_train):
        y_train_pred[:, i] = safe_predict_proba_class_1(est, X_train_proc)
    train_loss = competition_log_loss(y_train_filtered, y_train_pred)
    
    print(f"Final {cv_name} Train Loss: {train_loss:.4f}")






# submission generation
model_keys = list(final_pipelines.keys())

print("[NOTE] Generating a submission file for each trained model type.")
    
# Submission 1: Drug GroupKFold
submission_gkf = generate_submission(
        model_pipelines=[final_pipelines['Drug GroupKFold']], 
        X_test=X_test, 
        sample_submission=sample_submission, 
        TARGET_COLS=TARGET_COLS
    )
print("\n--- Submission 1 (Drug GroupKFold) ---")
print(f"Parameters: {best_params_all['Drug GroupKFold']}")
# submission_gkf.to_csv("submission_drug_gkf.csv", index=False)
    
# Submission 2: MoA StratifiedKFold (Binned)
submission_skf_moa = generate_submission(
        model_pipelines=[final_pipelines['MoA StratifiedKFold (Binned)']], 
        X_test=X_test, 
        sample_submission=sample_submission, 
        TARGET_COLS=TARGET_COLS
    )
print("\n--- Submission 2 (MoA StratifiedKFold Binned) ---")
print(f"Parameters: {best_params_all['MoA StratifiedKFold (Binned)']}")
# submission_skf_moa.to_csv("submission_moa_skf_binned.csv", index=False)

# Submission 3: Combined Drug+MoA Stratification
submission_combined = generate_submission(
        model_pipelines=[final_pipelines['Combined Drug+MoA Stratification']], 
        X_test=X_test, 
        sample_submission=sample_submission, 
        TARGET_COLS=TARGET_COLS
    )
print("\n--- Submission 3 (Combined Drug+MoA Stratification) ---")
print(f"Parameters: {best_params_all['Combined Drug+MoA Stratification']}")
# submission_combined.to_csv("submission_combined_strat.csv", index=False)


--- Data Loading and Preparation ---
Setting up 5-fold cross-validation...

======== Starting CV: Drug GroupKFold ========


KeyboardInterrupt: 

# Random Fores - Valid

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, StratifiedKFold 
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import log_loss
from itertools import product

# evaluation metric and loss function
def competition_log_loss(y_true, y_pred):
    eps = 1e-15
    y_pred = np.clip(y_pred, eps, 1 - eps)
    losses = []
    
    for i in range(y_true.shape[0]):
        loss = log_loss(y_true.iloc[i, :], y_pred[i, :], labels=[0, 1])
        losses.append(loss)
    
    return np.mean(losses)



def generate_submission(model_pipelines, X_test, sample_submission, TARGET_COLS):
    
    is_compound_test = X_test['cp_type'] != 'ctl_vehicle'
    X_test_cp = X_test[is_compound_test].drop(columns=['sig_id'])

    num_targets = len(TARGET_COLS)
    final_y_test_pred_sum = np.zeros((X_test_cp.shape[0], num_targets))
    
    for model_pipeline in model_pipelines:
        X_test_proc = model_pipeline.named_steps['preprocessor'].transform(X_test_cp)

        y_test_pred_cp = np.zeros((X_test_cp.shape[0], num_targets))
        classifier_estimators = model_pipeline.named_steps['classifier'].estimators_
        
        for i, est in enumerate(classifier_estimators):
            y_test_pred_cp[:, i] = safe_predict_proba_class_1(est, X_test_proc)
        
        final_y_test_pred_sum += y_test_pred_cp

    y_test_pred_final = final_y_test_pred_sum / len(model_pipelines)

    submission = sample_submission.copy()
    submission.loc[:, TARGET_COLS] = 0.0

    compound_sig_ids = X_test[is_compound_test]['sig_id'].values
    submission_indices = submission[submission['sig_id'].isin(compound_sig_ids)].index
    submission.loc[submission_indices, TARGET_COLS] = y_test_pred_final
    
    return submission















def safe_predict_proba_class_1(estimator, X):
    """
    Safely predicts probability of class 1 (p=1), handling degenerate classifiers 
    that were only trained on one class (e.g., all 0s in a sparse MoA target).
    """
    probas = estimator.predict_proba(X)
    
    # Check if the classifier produced two probability columns (binary case)
    if probas.shape[1] == 2:
        
        return probas[:, 1]
    else:
        # If probas.shape[1] == 1, the classifier is degenerate. 
        # Since MoA targets are sparse (mostly 0s), we assume p(class=1) is 0.0.
        return np.zeros(X.shape[0])



# Data Loading and Preparation
TARGET_COLS = [col for col in y_train.columns if col != 'sig_id']
ALL_FEATURES = X_train.columns.tolist()
NUM_FEATURES = [col for col in ALL_FEATURES if col.startswith('g-') or col.startswith('c-')]
CAT_FEATURES = ['cp_type', 'cp_time', 'cp_dose']
FEATURES = NUM_FEATURES + CAT_FEATURES


# Filter out control samples from the training set, as they have no MoAs.
is_compound_train = X_train['cp_type'] != 'ctl_vehicle'
X_train_filtered = X_train[is_compound_train].reset_index(drop=True)
y_train_filtered = y_train[is_compound_train].drop(columns=['sig_id']).reset_index(drop=True)

# save DRUG_ID and MOA_COUNT_TARGET and COMBINED_CV_TARGET  before dropping from features
DRUG_ID = X_train_filtered['drug_id']

X_train_filtered['moa_count'] = y_train_filtered.sum(axis=1)
MOA_COUNT_TARGET = X_train_filtered['moa_count']

X_train_filtered['combined_cv_target'] = DRUG_ID.astype(str) + '_' + MOA_COUNT_TARGET.astype(str)
COMBINED_CV_TARGET = X_train_filtered['combined_cv_target']

# Drop columns not needed for training features
X_train_filtered = X_train_filtered.drop(columns=['sig_id', 'moa_count', 'combined_cv_target', 'drug_id'])

# cross validation configuration
N_SPLITS = 5


# define CV strategies
gkf = GroupKFold(n_splits=N_SPLITS) # CV 1: by drug_id (pure grouping)
skf_moa = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42) # CV 2: by BINNED MoA count (pure stratification)
skf_combined = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42) # CV 3: by Combined Target (simulated combined CV)

cv_configs = [
    {'name': 'Drug GroupKFold', 'cv_object': gkf, 'groups': DRUG_ID, 'stratify_target': None},
    {'name': 'MoA StratifiedKFold (Binned)', 'cv_object': skf_moa, 'groups': None, 'stratify_target': MOA_COUNT_TARGET},
    {'name': 'Combined Drug+MoA Stratification', 'cv_object': skf_combined, 'groups': None, 'stratify_target': COMBINED_CV_TARGET}
]

# preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), NUM_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_FEATURES)
    ],
    remainder='drop' 
)

# versions of random forest
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, 15],
    'max_features': ['sqrt', 0.5]
}
param_names = list(param_grid.keys())
param_combinations = list(product(*param_grid.values()))

# store best parameters and losses from CV types
best_models = {}

# cross validation
for config in cv_configs:

    cv_name = config['name']
    cv_object = config['cv_object']
    groups = config['groups']
    stratify_target = config['stratify_target']
    
    
    current_best_loss = float('inf')
    current_best_params = None
    param_results = {}
    
    for params_tuple in param_combinations:
        current_params = dict(zip(param_names, params_tuple))
        
        fold_losses = []
        

        if cv_name == 'Drug GroupKFold':
            cv_splits = cv_object.split(X_train_filtered, y_train_filtered, groups=groups)
        else: 
            cv_splits = cv_object.split(X_train_filtered, stratify_target)


        # Iterate through cross-validation folds
        for fold, (train_index, val_index) in enumerate(cv_splits):
            
            # Split data for the current fold
            X_train_fold = X_train_filtered.iloc[train_index]
            X_valid_fold = X_train_filtered.iloc[val_index]
            y_train_fold = y_train_filtered.iloc[train_index]
            y_valid_fold = y_train_filtered.iloc[val_index]

            # Define model with current parameters
            current_tree = RandomForestClassifier(random_state=42, **current_params)
            current_multi_model = MultiOutputClassifier(current_tree, n_jobs=-1)
            #train
            current_pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor),
                ('classifier', current_multi_model)
            ])
            current_pipeline.fit(X_train_fold, y_train_fold)
            
            # Calculate Validation Loss
            X_valid_proc = current_pipeline.named_steps['preprocessor'].transform(X_valid_fold)
            y_valid_pred = np.zeros(y_valid_fold.shape)
            
            estimators_valid = current_pipeline.named_steps['classifier'].estimators_
            for i, est in enumerate(estimators_valid):
                y_valid_pred[:, i] = safe_predict_proba_class_1(est, X_valid_proc)
                
            val_loss = competition_log_loss(y_valid_fold, y_valid_pred)
            fold_losses.append(val_loss)

        # Calculate average loss for the current parameter set
        avg_val_loss = np.mean(fold_losses)
        param_results[params_tuple] = avg_val_loss
        
        # Check for best parameters for this CV type
        if avg_val_loss < current_best_loss:
            current_best_loss = avg_val_loss
            current_best_params = current_params

        print(f"  Model: {current_params} | Avg Val Loss: {avg_val_loss:.4f}")

    # Store and display the best result for this CV type
    best_models[cv_name] = {
        'best_params': current_best_params,
        'best_loss': current_best_loss
    }
    
    print(f"\n--- Best Model for {cv_name} ---")
    print(f"Best Loss: {current_best_loss:.4f}, Params: {current_best_params}")




# final Training on Full Data using the best models
final_pipelines = {}
best_params_all = {}

X_train_full = X_train_filtered

for cv_name, result in best_models.items():
    best_params = result['best_params']
    
    print(f"\nTraining FINAL model for {cv_name} on ALL training data...")
    
    # Create Model using best parameters
    final_tree = RandomForestClassifier(random_state=42, **best_params)
    final_multi_model = MultiOutputClassifier(final_tree, n_jobs=-1)

    # Create Final Pipeline
    final_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', final_multi_model)
    ])

    # Train on ALL available filtered data
    final_pipeline.fit(X_train_full, y_train_filtered)
    
    # Store the pipeline and parameters 
    final_pipelines[cv_name] = final_pipeline
    best_params_all[cv_name] = best_params
    
    # Calculate Training Loss for reference
    X_train_proc = final_pipeline.named_steps['preprocessor'].transform(X_train_full)
    y_train_pred = np.zeros(y_train_filtered.shape)
    

    estimators_train = final_pipeline.named_steps['classifier'].estimators_
    for i, est in enumerate(estimators_train):
        y_train_pred[:, i] = safe_predict_proba_class_1(est, X_train_proc)
    train_loss = competition_log_loss(y_train_filtered, y_train_pred)
    
    print(f"Final {cv_name} Train Loss: {train_loss:.4f}")






# submission generation
model_keys = list(final_pipelines.keys())

print("[NOTE] Generating a submission file for each trained model type.")
    
# Submission 1: Drug GroupKFold
submission_gkf = generate_submission(
        model_pipelines=[final_pipelines['Drug GroupKFold']], 
        X_test=X_test, 
        sample_submission=sample_submission, 
        TARGET_COLS=TARGET_COLS
    )
print("\n--- Submission 1 (Drug GroupKFold) ---")
print(f"Parameters: {best_params_all['Drug GroupKFold']}")
# submission_gkf.to_csv("submission_drug_gkf.csv", index=False)
    
# Submission 2: MoA StratifiedKFold (Binned)
submission_skf_moa = generate_submission(
        model_pipelines=[final_pipelines['MoA StratifiedKFold (Binned)']], 
        X_test=X_test, 
        sample_submission=sample_submission, 
        TARGET_COLS=TARGET_COLS
    )
print("\n--- Submission 2 (MoA StratifiedKFold Binned) ---")
print(f"Parameters: {best_params_all['MoA StratifiedKFold (Binned)']}")
# submission_skf_moa.to_csv("submission_moa_skf_binned.csv", index=False)

# Submission 3: Combined Drug+MoA Stratification
submission_combined = generate_submission(
        model_pipelines=[final_pipelines['Combined Drug+MoA Stratification']], 
        X_test=X_test, 
        sample_submission=sample_submission, 
        TARGET_COLS=TARGET_COLS
    )
print("\n--- Submission 3 (Combined Drug+MoA Stratification) ---")
print(f"Parameters: {best_params_all['Combined Drug+MoA Stratification']}")
# submission_combined.to_csv("submission_combined_strat.csv", index=False)


# xgboost - Valid

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, StratifiedKFold 
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import log_loss
from itertools import product
import xgboost as xgb

# evaluation metric and loss function
def competition_log_loss(y_true, y_pred):
    eps = 1e-15
    y_pred = np.clip(y_pred, eps, 1 - eps)
    losses = []
    
    for i in range(y_true.shape[0]):
        loss = log_loss(y_true.iloc[i, :], y_pred[i, :], labels=[0, 1])
        losses.append(loss)
    
    return np.mean(losses)



def generate_submission(model_pipelines, X_test, sample_submission, TARGET_COLS):
    
    is_compound_test = X_test['cp_type'] != 'ctl_vehicle'
    X_test_cp = X_test[is_compound_test].drop(columns=['sig_id'])

    num_targets = len(TARGET_COLS)
    final_y_test_pred_sum = np.zeros((X_test_cp.shape[0], num_targets))
    
    for model_pipeline in model_pipelines:
        X_test_proc = model_pipeline.named_steps['preprocessor'].transform(X_test_cp)

        y_test_pred_cp = np.zeros((X_test_cp.shape[0], num_targets))
        classifier_estimators = model_pipeline.named_steps['classifier'].estimators_
        
        for i, est in enumerate(classifier_estimators):
            y_test_pred_cp[:, i] = safe_predict_proba_class_1(est, X_test_proc)
        
        final_y_test_pred_sum += y_test_pred_cp

    y_test_pred_final = final_y_test_pred_sum / len(model_pipelines)

    submission = sample_submission.copy()
    submission.loc[:, TARGET_COLS] = 0.0

    compound_sig_ids = X_test[is_compound_test]['sig_id'].values
    submission_indices = submission[submission['sig_id'].isin(compound_sig_ids)].index
    submission.loc[submission_indices, TARGET_COLS] = y_test_pred_final
    
    return submission















def safe_predict_proba_class_1(estimator, X):
    """
    Safely predicts probability of class 1 (p=1), handling degenerate classifiers 
    that were only trained on one class (e.g., all 0s in a sparse MoA target).
    """
    probas = estimator.predict_proba(X)
    
    # Check if the classifier produced two probability columns (binary case)
    if probas.shape[1] == 2:
        
        return probas[:, 1]
    else:
        # If probas.shape[1] == 1, the classifier is degenerate. 
        # Since MoA targets are sparse (mostly 0s), we assume p(class=1) is 0.0.
        return np.zeros(X.shape[0])



# Data Loading and Preparation
TARGET_COLS = [col for col in y_train.columns if col != 'sig_id']
ALL_FEATURES = X_train.columns.tolist()
NUM_FEATURES = [col for col in ALL_FEATURES if col.startswith('g-') or col.startswith('c-')]
CAT_FEATURES = ['cp_type', 'cp_time', 'cp_dose']
FEATURES = NUM_FEATURES + CAT_FEATURES


# Filter out control samples from the training set, as they have no MoAs.
is_compound_train = X_train['cp_type'] != 'ctl_vehicle'
X_train_filtered = X_train[is_compound_train].reset_index(drop=True)
y_train_filtered = y_train[is_compound_train].drop(columns=['sig_id']).reset_index(drop=True)

# save DRUG_ID and MOA_COUNT_TARGET and COMBINED_CV_TARGET  before dropping from features
DRUG_ID = X_train_filtered['drug_id']

X_train_filtered['moa_count'] = y_train_filtered.sum(axis=1)
MOA_COUNT_TARGET = X_train_filtered['moa_count']

X_train_filtered['combined_cv_target'] = DRUG_ID.astype(str) + '_' + MOA_COUNT_TARGET.astype(str)
COMBINED_CV_TARGET = X_train_filtered['combined_cv_target']

# Drop columns not needed for training features
X_train_filtered = X_train_filtered.drop(columns=['sig_id', 'moa_count', 'combined_cv_target', 'drug_id'])

# cross validation configuration
N_SPLITS = 5


# define CV strategies
gkf = GroupKFold(n_splits=N_SPLITS) # CV 1: by drug_id (pure grouping)
skf_moa = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42) # CV 2: by BINNED MoA count (pure stratification)
skf_combined = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42) # CV 3: by Combined Target (simulated combined CV)

cv_configs = [
    {'name': 'Drug GroupKFold', 'cv_object': gkf, 'groups': DRUG_ID, 'stratify_target': None},
    {'name': 'MoA StratifiedKFold (Binned)', 'cv_object': skf_moa, 'groups': None, 'stratify_target': MOA_COUNT_TARGET},
    {'name': 'Combined Drug+MoA Stratification', 'cv_object': skf_combined, 'groups': None, 'stratify_target': COMBINED_CV_TARGET}
]

# preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), NUM_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_FEATURES)
    ],
    remainder='drop' 
)


# versions of xgboost with different params
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.7]
}

param_names = list(param_grid.keys())
param_combinations = list(product(*param_grid.values()))

# store best parameters and losses from CV types
best_models = {}

# cross validation
for config in cv_configs:

    cv_name = config['name']
    cv_object = config['cv_object']
    groups = config['groups']
    stratify_target = config['stratify_target']
    
    
    current_best_loss = float('inf')
    current_best_params = None
    param_results = {}
    
    for params_tuple in param_combinations:
        current_params = dict(zip(param_names, params_tuple))
        
        fold_losses = []
        

        if cv_name == 'Drug GroupKFold':
            cv_splits = cv_object.split(X_train_filtered, y_train_filtered, groups=groups)
        else: 
            cv_splits = cv_object.split(X_train_filtered, stratify_target)


        # Iterate through cross-validation folds
        for fold, (train_index, val_index) in enumerate(cv_splits):
            
            # Split data for the current fold
            X_train_fold = X_train_filtered.iloc[train_index]
            X_valid_fold = X_train_filtered.iloc[val_index]
            y_train_fold = y_train_filtered.iloc[train_index]
            y_valid_fold = y_train_filtered.iloc[val_index]

            # Define model with current parameters
            current_tree = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='gpu_hist',
        use_label_encoder=False,
        random_state=42, 
        n_jobs=-1, 
        **current_params
    )
        
            current_multi_model = MultiOutputClassifier(current_tree, n_jobs=-1)
            #train
            current_pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor),
                ('classifier', current_multi_model)
            ])
            current_pipeline.fit(X_train_fold, y_train_fold)
            
            # Calculate Validation Loss
            X_valid_proc = current_pipeline.named_steps['preprocessor'].transform(X_valid_fold)
            y_valid_pred = np.zeros(y_valid_fold.shape)
            
            estimators_valid = current_pipeline.named_steps['classifier'].estimators_
            for i, est in enumerate(estimators_valid):
                y_valid_pred[:, i] = safe_predict_proba_class_1(est, X_valid_proc)
                
            val_loss = competition_log_loss(y_valid_fold, y_valid_pred)
            fold_losses.append(val_loss)

        # Calculate average loss for the current parameter set
        avg_val_loss = np.mean(fold_losses)
        param_results[params_tuple] = avg_val_loss
        
        # Check for best parameters for this CV type
        if avg_val_loss < current_best_loss:
            current_best_loss = avg_val_loss
            current_best_params = current_params

        print(f"  Model: {current_params} | Avg Val Loss: {avg_val_loss:.4f}")

    # Store and display the best result for this CV type
    best_models[cv_name] = {
        'best_params': current_best_params,
        'best_loss': current_best_loss
    }
    
    print(f"\n--- Best Model for {cv_name} ---")
    print(f"Best Loss: {current_best_loss:.4f}, Params: {current_best_params}")




# final Training on Full Data using the best models
final_pipelines = {}
best_params_all = {}

X_train_full = X_train_filtered

for cv_name, result in best_models.items():
    best_params = result['best_params']
    
    print(f"\nTraining FINAL model for {cv_name} on ALL training data...")
    
    # Create Model using best parameters
    current_tree = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='gpu_hist',
        use_label_encoder=False,
        random_state=42, 
        n_jobs=-1, 
        **current_params
    )
    final_multi_model = MultiOutputClassifier(final_tree, n_jobs=-1)

    # Create Final Pipeline
    final_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', final_multi_model)
    ])

    # Train on ALL available filtered data
    final_pipeline.fit(X_train_full, y_train_filtered)
    
    # Store the pipeline and parameters 
    final_pipelines[cv_name] = final_pipeline
    best_params_all[cv_name] = best_params
    
    # Calculate Training Loss for reference
    X_train_proc = final_pipeline.named_steps['preprocessor'].transform(X_train_full)
    y_train_pred = np.zeros(y_train_filtered.shape)
    

    estimators_train = final_pipeline.named_steps['classifier'].estimators_
    for i, est in enumerate(estimators_train):
        y_train_pred[:, i] = safe_predict_proba_class_1(est, X_train_proc)
    train_loss = competition_log_loss(y_train_filtered, y_train_pred)
    
    print(f"Final {cv_name} Train Loss: {train_loss:.4f}")






# submission generation
model_keys = list(final_pipelines.keys())

print("[NOTE] Generating a submission file for each trained model type.")
    
# Submission 1: Drug GroupKFold
submission_gkf = generate_submission(
        model_pipelines=[final_pipelines['Drug GroupKFold']], 
        X_test=X_test, 
        sample_submission=sample_submission, 
        TARGET_COLS=TARGET_COLS
    )
print("\n--- Submission 1 (Drug GroupKFold) ---")
print(f"Parameters: {best_params_all['Drug GroupKFold']}")
# submission_gkf.to_csv("submission_drug_gkf.csv", index=False)
    
# Submission 2: MoA StratifiedKFold (Binned)
submission_skf_moa = generate_submission(
        model_pipelines=[final_pipelines['MoA StratifiedKFold (Binned)']], 
        X_test=X_test, 
        sample_submission=sample_submission, 
        TARGET_COLS=TARGET_COLS
    )
print("\n--- Submission 2 (MoA StratifiedKFold Binned) ---")
print(f"Parameters: {best_params_all['MoA StratifiedKFold (Binned)']}")
# submission_skf_moa.to_csv("submission_moa_skf_binned.csv", index=False)

# Submission 3: Combined Drug+MoA Stratification
submission_combined = generate_submission(
        model_pipelines=[final_pipelines['Combined Drug+MoA Stratification']], 
        X_test=X_test, 
        sample_submission=sample_submission, 
        TARGET_COLS=TARGET_COLS
    )
print("\n--- Submission 3 (Combined Drug+MoA Stratification) ---")
print(f"Parameters: {best_params_all['Combined Drug+MoA Stratification']}")
# submission_combined.to_csv("submission_combined_strat.csv", index=False)
